# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassaanSaqib/FlyRankAI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
# Setup: connect DuckDB to the gated Hugging Face warehouse

%pip -q install duckdb

import duckdb

from google.colab import userdata

# Read the token securely from Colab Secrets

HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection

con = duckdb.connect()

# Register the Hugging Face token securely

con.execute(f"""

    CREATE OR REPLACE SECRET hf (

        TYPE huggingface,

        TOKEN '{HF_TOKEN}'

    )

""")

# March 2026 partition: use a mid-panel month for development

REL = """

read_parquet(

    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'

)

"""

print("✅ DuckDB connected to the March 2026 warehouse partition.")

✅ DuckDB connected to the March 2026 warehouse partition.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row in my analysis represents one pseudonymized content item for one client on one report date.
Table: I use fact_content_daily_performance as the main table for my Refresh / Content Opportunity Scoring lane.
Time window: I develop and verify the contract using March 2026 (2026-03-01 to 2026-03-31) as a mid-panel month. I do not use the June 2026 sample for label development because it is the final month and should remain a sealed test window.
Prediction/ranking goal: Rank content items by refresh opportunity so a reviewer can prioritize which pages deserve attention first.
Deliberate exclusion: I exclude future or label-derived information from the feature set because it would leak the answer into the model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the intended March 2026 analysis window.

section1_check = con.execute(f"""

SELECT

    MIN(report_date) AS start_date,

    MAX(report_date) AS end_date,

    COUNT(*) AS total_rows

FROM {REL}

""").df()

section1_check

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields_check = con.execute(f"""

SELECT

    report_date,

    client_hash_id,

    content_hash_id,

    gsc_data_available,

    ga4_data_available,

    gsc_impressions,

    gsc_clicks,

    CASE

        WHEN gsc_impressions > 0

        THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions

        ELSE 0

    END AS gsc_ctr,

    gsc_avg_position,

    ga4_sessions

FROM {REL}

LIMIT 10

""").df()

fields_check

1. Search impressions — how often the content appeared in search results.
2. Search clicks — how many visits came from search clicks.
3. Click-through rate (CTR) — the relationship between impressions and clicks.
4. Average search position — the content’s observed ranking position in search results.
5. Analytics traffic/sessions — an available measure of user visits to the content.

**Label / proxy:** My target is a refresh-opportunity ranking rather than a claim that a page definitely needs updating. I will use a future-period performance outcome or change as the proxy for whether a content item was a useful refresh opportunity.

**Context fields:** Report date, pseudonymized client identifier, pseudonymized content identifier, and the relevant data-availability flags. These fields identify the observation and determine whether the required signals were actually available.

**Excluded:** I deliberately exclude any future-period performance values used to create the label/proxy from the honest feature set. They would not be known at the decision moment and would create target leakage. I also exclude private client names, URLs, and raw identifying information.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

# ============================================================

# VERIFICATION QUERY 1 — GRAIN

# One row should be unique by:

# report_date + client_hash_id + content_hash_id

# ============================================================

grain_check = con.execute(f"""

SELECT

    COUNT(*) AS total_rows,

    COUNT(DISTINCT (

        report_date,

        client_hash_id,

        content_hash_id

    )) AS unique_grain_rows,

    COUNT(*) - COUNT(DISTINCT (

        report_date,

        client_hash_id,

        content_hash_id

    )) AS duplicate_rows

FROM {REL}

""").df()

print("VERIFICATION QUERY 1 — GRAIN")

display(grain_check)

# ============================================================

# VERIFICATION QUERY 2 — ROW COUNT + DATE SPAN

# ============================================================

slice_check = con.execute(f"""

SELECT

    COUNT(*) AS row_count,

    MIN(report_date) AS min_date,

    MAX(report_date) AS max_date

FROM {REL}

""").df()

print("\nVERIFICATION QUERY 2 — SLICE SIZE AND DATE SPAN")

display(slice_check)

# ============================================================

# VERIFICATION QUERY 3 — AVAILABILITY

# Required use of IS TRUE

# Keep rows where both GSC and GA4 data are available

# ============================================================

availability_check = con.execute(f"""

SELECT

    COUNT(*) AS total_rows,

    COUNT(*) FILTER (

        WHERE gsc_data_available IS TRUE

          AND ga4_data_available IS TRUE

    ) AS available_rows

FROM {REL}

""").df()

print("\nVERIFICATION QUERY 3 — DATA AVAILABILITY")

display(availability_check)

# ============================================================

# FIVE-FEATURE FRAME

#

# Decision period: March 1–21, 2026

# Future outcome period: March 22–31, 2026

#

# Features:

# 1. gsc_impressions

# 2. gsc_clicks

# 3. gsc_ctr

# 4. gsc_avg_position

# 5. ga4_sessions

#

# Label proxy:

# Whether future-period clicks are above the median.

# ============================================================

feature_frame = con.execute(f"""

WITH decision_period AS (

    SELECT

        client_hash_id,

        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,

        SUM(gsc_clicks) AS gsc_clicks,

        CASE

            WHEN SUM(gsc_impressions) > 0

            THEN CAST(SUM(gsc_clicks) AS DOUBLE)

                 / SUM(gsc_impressions)

            ELSE 0

        END AS gsc_ctr,

        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions

    FROM {REL}

    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'

      AND gsc_data_available IS TRUE

      AND ga4_data_available IS TRUE

    GROUP BY

        client_hash_id,

        content_hash_id

),

future_period AS (

    SELECT

        client_hash_id,

        content_hash_id,

        SUM(gsc_clicks) AS future_clicks

    FROM {REL}

    WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'

      AND gsc_data_available IS TRUE

    GROUP BY

        client_hash_id,

        content_hash_id

)

SELECT

    d.client_hash_id,

    d.content_hash_id,

    d.gsc_impressions,

    d.gsc_clicks,

    d.gsc_ctr,

    d.gsc_avg_position,

    d.ga4_sessions,

    COALESCE(f.future_clicks, 0) AS future_clicks

FROM decision_period d

LEFT JOIN future_period f

    USING (client_hash_id, content_hash_id)

LIMIT 50000

""").df()

# Create a simple binary proxy label:

# 1 = future clicks above the median

# 0 = future clicks at or below the median

future_median = feature_frame["future_clicks"].median()

feature_frame["refresh_opportunity_label"] = (

    feature_frame["future_clicks"] > future_median

).astype(int)

print("\nFIVE-FEATURE FRAME — SAMPLE")

display(

    feature_frame[

        [

            "gsc_impressions",

            "gsc_clicks",

            "gsc_ctr",

            "gsc_avg_position",

            "ga4_sessions",

            "refresh_opportunity_label"

        ]

    ].head(10)

)

# ============================================================

# DELIBERATE LEAKAGE EXPERIMENT

# ============================================================

feature_columns = [

    "gsc_impressions",

    "gsc_clicks",

    "gsc_ctr",

    "gsc_avg_position",

    "ga4_sessions"

]

X = feature_frame[feature_columns].fillna(0)

y = feature_frame["refresh_opportunity_label"]

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.25,

    random_state=42,

    stratify=y

)

# Honest model — only information available at decision time

honest_model = DecisionTreeClassifier(

    max_depth=5,

    random_state=42

)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_accuracy = accuracy_score(

    y_test,

    honest_predictions

)

# ------------------------------------------------------------

# LEAKAGE TRAP

#

# Add a label-derived column ON PURPOSE.

# This column directly reveals the answer.

# ------------------------------------------------------------

leaky_frame = feature_frame.copy()

leaky_frame["LEAKED_LABEL_FEATURE"] = (

    leaky_frame["refresh_opportunity_label"]

)

X_leaky = leaky_frame[

    feature_columns + ["LEAKED_LABEL_FEATURE"]

].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(

    X_leaky,

    y,

    test_size=0.25,

    random_state=42,

    stratify=y

)

leaky_model = DecisionTreeClassifier(

    max_depth=5,

    random_state=42

)

leaky_model.fit(X_train_l, y_train_l)

leaky_predictions = leaky_model.predict(X_test_l)

leaky_accuracy = accuracy_score(

    y_test_l,

    leaky_predictions

)

print("\nLEAKAGE EXPERIMENT")

print(f"Honest accuracy: {honest_accuracy:.4f}")

print(f"Leaky accuracy:  {leaky_accuracy:.4f}")

print(

    "\nThe leaked feature directly contains information derived "

    "from the label, so the score becomes unrealistically high."

)

print(

    "The leaked column is removed from the final feature set. "

    "The honest accuracy is the result I keep."

)

# Final honest feature list

final_features = feature_frame[feature_columns].copy()

print("\nFINAL HONEST FEATURE SET")

display(final_features.head())

I verify the contract with exactly three queries using the March 2026 development window:

**Query 1 — Grain**: Check whether each combination of report date, pseudonymized client, and pseudonymized content item is unique. This verifies that one row represents one content item for one client on one date.

**Query 2 — Slice size and date span:** Count the rows in my March 2026 slice and report the minimum and maximum report dates. This confirms how much data is in the development slice and that the intended time window is being used.

**Query 3 — Availability:** Filter the relevant availability flag with IS TRUE and count how many rows remain. This prevents unavailable or structurally missing data from being treated as real zero-valued observations.

**Five-feature frame:** I build a small feature frame using only the five selected signals: impressions, clicks, CTR, average search position, and traffic/sessions.

**Available when?**

**Impressions:** Knowable at the decision moment because they describe search exposure already observed up to that date.
Clicks: Knowable at the decision moment because they are historical search interactions already recorded.
CTR: Knowable at the decision moment because it is calculated from already observed impressions and clicks.
Average search position: Knowable at the decision moment because it reflects ranking information already observed for the content.
Traffic/sessions: Knowable at the decision moment because I use only analytics observations available up to that point and only where the relevant availability flag is true.
Deliberate leakage experiment: I intentionally add one feature derived directly from the future outcome used to construct the label. I expect the quick validation score to increase unrealistically because the model is being given information about the answer. I then remove this leaked feature and keep the lower, honest score as the valid result. The purpose of this experiment is to demonstrate why feature availability at the decision moment matters

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
coverage_check = con.execute(f"""

SELECT

    client_hash_id,

    MIN(report_date) AS first_date,

    MAX(report_date) AS last_date,

    COUNT(DISTINCT report_date) AS observed_days

FROM {REL}

GROUP BY client_hash_id

ORDER BY observed_days ASC

LIMIT 10

""").df()

print("Example of unbalanced client history coverage:")

display(coverage_check)

print(

    "\nLimitation: clients can have different amounts of observed history, "

    "so comparisons should be treated as directional decision support rather "

    "than proof that a refresh causes better future performance."

)

**Limitation**: This slice cannot prove that refreshing a content item causes future performance to improve. The warehouse contains observational performance data, and clients may have different amounts of historical coverage. Search and analytics signals may also become available at different times, so I must respect the availability flags and avoid treating unavailable historical data as genuine zero performance. The resulting refresh-opportunity score should therefore be interpreted as a directional decision-support ranking, not as a guaranteed prediction of the effect of refreshing a page

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.